# Advanced KPIs · coverage and source reconciliation

**Optional extension; does not change core scores or ranks.** Input: core_analysis.pkl and cached raw exports. Output: advanced_analysis.pkl with the six provider metrics, provenance and coverage-aware summaries. Covered-period rates describe a selected subset and must not be presented as complete-season estimates.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [1]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_context("core_analysis"))
player_kpis_data, _ = load_or_download_json(
    f"{IMPECT_BASE_URL}/player_kpis/player_kpis_{example_match_id}.json",
    RAW_DIR / "player_kpis" / f"player_kpis_{example_match_id}.json")


Project: /workspace/scratch/491a21a4af19/organized_project/football-no8-recruitment


## Review available season fields

The core event table intentionally selects fields for this analysis; it does not contain every field offered by IMPECT.

In [2]:
# See every available column name.
print("\n".join(season_events.columns))

# See the first 20 event records, with all columns visible.
with pd.option_context("display.max_columns", None):
    display(season_events.head(20))

match_id
id
index
periodId
gameTime.gameTimeInSec
squadId
player.id
player.position
actionType
action
result
phase
pressure
start.adjCoordinates.x
start.adjCoordinates.y
end.adjCoordinates.x
end.adjCoordinates.y
    match_id          id  index  periodId  gameTime.gameTimeInSec  squadId  \
0     122838  4858179098      0         1                  0.0000     33.0   
1     122838  4858179099      1         1                  0.3320      NaN   
2     122838  4858179100      2         1                  1.6510     33.0   
3     122838  4858179101      3         1                  1.6511     33.0   
4     122838  4858179102      4         1                  4.1930     33.0   
5     122838  4858179103      5         1                  6.9050     38.0   
6     122838  4858179104      6         1                  6.9051     38.0   
7     122838  4858179105      7         1                  8.2600     38.0   
8     122838  4858179106      8         1                  8.2601     38.0   
9     12

## Inspect supplied advanced values

Display example match xG and bypassed-opponent values. These provider KPIs are not newly calculated models.

In [3]:
rows = []

for side in ["squadHome", "squadAway"]:
    squad = player_kpis_data[side]

    for player in squad["players"]:
        kpis = {
            item["kpiId"]: item["value"]
            for item in player["kpis"]
        }

        rows.append({
            "player.id": player["id"],
            "position": player["position"],
            "xG": kpis.get(82),
            "bypassed_opponents": kpis.get(0),
        })

match_stats = pd.DataFrame(rows).merge(
    player_lookup,
    on="player.id",
    how="left",
    validate="many_to_one",
)

display(
    match_stats[
        ["player_name", "position", "xG", "bypassed_opponents"]
    ].sort_values("xG", ascending=False)
)

                 player_name  ... bypassed_opponents
17                Leroy Sané  ...              23.80
19                Harry Kane  ...              20.80
27         Noussair Mazraoui  ...              79.10
2       Leonardo Bittencourt  ...              17.90
11           Niclas Füllkrug  ...               8.10
30             Jamal Musiala  ...              23.30
22             Leon Goretzka  ...              30.90
20            Kingsley Coman  ...              12.90
13                Jens Stage  ...               5.00
0            Mitchell Weiser  ...               6.50
31                Mathys Tel  ...               2.50
16            Joshua Kimmich  ...              76.40
12            Marvin Ducksch  ...               7.92
29               Min-jae Kim  ...              79.70
4              Romano Schmid  ...               1.20
1            Milos Veljkovic  ...              27.10
3               Anthony Jung  ...              29.58
5               Marco Friedl  ...             

## Inspect the four pxT KPIs

Read provider-supplied passing, carrying, set-piece and shooting pxT values; preserve missing and negative values.

In [4]:
pxt_metrics = {
    1404: "PXT_PASS",
    1405: "PXT_DRIBBLE",
    1406: "PXT_SETPIECE",
    1408: "PXT_SHOT",
}

rows = []

for side in ["squadHome", "squadAway"]:
    squad = player_kpis_data[side]

    for player in squad["players"]:
        # Match each KPI ID to its supplied value.
        kpis = {
            item["kpiId"]: item["value"]
            for item in player["kpis"]
        }

        row = {
            "match_id": player_kpis_data["matchId"],
            "squadId": squad["id"],
            "player.id": player["id"],
            "position": player["position"],
        }

        for kpi_id, metric_name in pxt_metrics.items():
            row[metric_name] = kpis.get(kpi_id)

        rows.append(row)

# Attach player and team names using their IDs.
match_pxt = (
    pd.DataFrame(rows)
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        squad_lookup,
        on="squadId",
        how="left",
        validate="many_to_one",
    )
)

print("Match ID:", player_kpis_data["matchId"])

display(
    match_pxt[
        [
            "player_name",
            "team_name",
            "position",
            "PXT_PASS",
            "PXT_DRIBBLE",
            "PXT_SETPIECE",
            "PXT_SHOT",
        ]
    ]
    .sort_values("PXT_PASS", ascending=False)
    .round(4)
)

Match ID: 122838
                 player_name          team_name  ... PXT_SETPIECE  PXT_SHOT
18             Thomas Müller  FC Bayern München  ...          NaN       NaN
27         Noussair Mazraoui  FC Bayern München  ...       0.0230    0.1780
21  Eric-Maxim Choupo-Moting  FC Bayern München  ...          NaN       NaN
28           Alphonso Davies  FC Bayern München  ...       0.0049       NaN
11           Niclas Füllkrug   SV Werder Bremen  ...      -0.0026   -0.0754
29               Min-jae Kim  FC Bayern München  ...       0.0003    0.1498
16            Joshua Kimmich  FC Bayern München  ...      -0.1011   -0.0598
2       Leonardo Bittencourt   SV Werder Bremen  ...          NaN   -0.1855
25          Matthijs de Ligt  FC Bayern München  ...          NaN       NaN
26           Dayot Upamecano  FC Bayern München  ...          NaN       NaN
30             Jamal Musiala  FC Bayern München  ...          NaN   -0.0570
5               Marco Friedl   SV Werder Bremen  ...          NaN      

## Audit six advanced KPIs

Extract six metrics at the same player-position-match grain as the duration records and quantify coverage.

In [5]:
# 1. The provider KPIs we want to investigate.
advanced_kpi_map = {
    0: "bypassed_opponents",
    1404: "pxt_pass",
    1405: "pxt_dribble",
    1406: "pxt_setpiece",
    1408: "pxt_shot",
    1782: "expected_goal_assists",
}

advanced_candidate_ids = set(midfield_pool["player.id"])
advanced_rows = []

# 2. Extract candidate midfield records from every match.
for match_id in season_match_ids:
    path = (
        RAW_DIR / "player_kpis"
        / f"player_kpis_{match_id}.json"
    )

    with path.open("r", encoding="utf-8") as file:
        match_data = json.load(file)

    for side in ["squadHome", "squadAway"]:
        squad = match_data[side]

        for player in squad["players"]:
            if (
                player["id"] not in advanced_candidate_ids
                or player["position"] not in midfield_positions
            ):
                continue

            values = {
                item["kpiId"]: item["value"]
                for item in player["kpis"]
            }

            row = {
                "match_id": match_id,
                "squadId": squad["id"],
                "player.id": player["id"],
                "player.position": player["position"],
            }

            for kpi_id, name in advanced_kpi_map.items():
                row[name] = values.get(kpi_id)

            advanced_rows.append(row)

advanced_records = pd.DataFrame(advanced_rows)

# 3. Verify that the extracted records match playing-time records.
advanced_keys = [
    "match_id", "squadId", "player.id", "player.position"
]

expected_advanced_records = season_durations.loc[
    season_durations["player.id"].isin(advanced_candidate_ids)
    & season_durations["player.position"].isin(midfield_positions),
    advanced_keys + ["minutes_played"],
]

advanced_records = expected_advanced_records.merge(
    advanced_records,
    on=advanced_keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not advanced_records["_merge"].eq("both").all():
    raise ValueError("KPI records do not match playing-time records.")

# 4. Measure coverage without replacing missing values with zero.
advanced_coverage_rows = []

for metric in advanced_kpi_map.values():
    present = advanced_records[metric].notna()

    complete_players = (
        present.groupby(advanced_records["player.id"]).all().sum()
    )

    advanced_coverage_rows.append({
        "metric": metric,
        "populated_records": int(present.sum()),
        "missing_records": int((~present).sum()),
        "record_coverage_pct": present.mean() * 100,
        "minutes_coverage_pct": (
            advanced_records.loc[present, "minutes_played"].sum()
            / advanced_records["minutes_played"].sum()
            * 100
        ),
        "players_complete": int(complete_players),
        "explicit_zeros": int(advanced_records[metric].eq(0).sum()),
        "negative_values": int(advanced_records[metric].lt(0).sum()),
    })

advanced_coverage = pd.DataFrame(advanced_coverage_rows)

print("Candidates:", advanced_records["player.id"].nunique())
print("Candidate midfield records:", len(advanced_records))
display(advanced_coverage.round(2))

Candidates: 79
Candidate midfield records: 2115
                  metric  populated_records  ...  explicit_zeros  negative_values
0     bypassed_opponents               2016  ...               0                0
1               pxt_pass               2099  ...               0              798
2            pxt_dribble               2065  ...               0             1425
3           pxt_setpiece               1081  ...               0              790
4               pxt_shot               1084  ...               0              638
5  expected_goal_assists               2015  ...               0                0

[6 rows x 8 columns]


## Compare missing KPIs with recorded actions

Distinguish absent actions, present actions and unmatched event-position records. None alone proves a missing KPI should be zero.

In [6]:
# Count recorded actions at the SAME grain as the KPI records:
# match + team + player + position.
activity_counts = (
    season_events.loc[
        season_events["player.id"].isin(
            advanced_records["player.id"].unique()
        )
        & season_events["player.position"].isin(midfield_positions)
    ]
    .groupby(advanced_keys + ["actionType"])
    .size()
    .unstack("actionType", fill_value=0)
    .reindex(columns=["PASS", "DRIBBLE", "SHOT"], fill_value=0)
    .rename(columns={
        "PASS": "recorded_passes",
        "DRIBBLE": "recorded_carries",
        "SHOT": "recorded_shots",
    })
    .reset_index()
)

activity_counts.columns.name = None

advanced_activity_records = (
    advanced_records.drop(columns="_merge", errors="ignore")
    .merge(
        activity_counts,
        on=advanced_keys,
        how="left",
        validate="one_to_one",
        indicator="event_match",
    )
)

# Do not turn unmatched event records into assumed zero actions.
activity_checks = {
    "pxt_pass": "recorded_passes",
    "pxt_dribble": "recorded_carries",
    "pxt_shot": "recorded_shots",
}

activity_audit_rows = []

for metric, action_column in activity_checks.items():
    missing_kpi = advanced_activity_records[metric].isna()
    matched_events = advanced_activity_records["event_match"].eq("both")

    activity_audit_rows.append({
        "metric": metric,
        "missing_kpi_records": int(missing_kpi.sum()),
        "missing_kpi_no_recorded_action": int((
            missing_kpi
            & matched_events
            & advanced_activity_records[action_column].eq(0)
        ).sum()),
        "missing_kpi_despite_recorded_action": int((
            missing_kpi
            & matched_events
            & advanced_activity_records[action_column].gt(0)
        ).sum()),
        "missing_kpi_no_matching_event_record": int((
            missing_kpi & ~matched_events
        ).sum()),
    })

activity_audit = pd.DataFrame(activity_audit_rows)
display(activity_audit)

        metric  ...  missing_kpi_no_matching_event_record
0     pxt_pass  ...                                    11
1  pxt_dribble  ...                                    11
2     pxt_shot  ...                                    11

[3 rows x 5 columns]


## Inspect unmatched position records

Look for events at another recorded position, without reassigning time or changing provider labels.

In [7]:
# Playing-time/KPI records without a matching
# match + team + player + position in the event table.
unmatched_position_records = advanced_activity_records.loc[
    advanced_activity_records["event_match"].eq("left_only"),
    advanced_keys + ["minutes_played"],
].copy()

# Check whether those players have events in the same match/team,
# possibly recorded under a DIFFERENT position.
player_match_keys = ["match_id", "squadId", "player.id"]

player_match_event_summary = (
    season_events.loc[
        season_events["player.id"].isin(
            unmatched_position_records["player.id"].unique()
        )
    ]
    .groupby(player_match_keys)
    .agg(
        recorded_events=("id", "size"),
        positions_in_events=(
            "player.position",
            lambda positions: " / ".join(
                sorted(positions.dropna().astype(str).unique())
            ),
        ),
    )
    .reset_index()
)

unmatched_record_review = (
    unmatched_position_records
    .merge(
        player_match_event_summary,
        on=player_match_keys,
        how="left",
        validate="many_to_one",
    )
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

unmatched_record_review["recorded_events"] = (
    unmatched_record_review["recorded_events"]
    .fillna(0)
    .astype(int)
)

print("Unmatched player-position records:",
      len(unmatched_record_review))

display(
    unmatched_record_review[
        [
            "match_id",
            "player_name",
            "player.position",
            "minutes_played",
            "recorded_events",
            "positions_in_events",
        ]
    ]
    .sort_values(["match_id", "player_name"])
    .reset_index(drop=True)
)

Unmatched player-position records: 11
    match_id  ...                    positions_in_events
0     122873  ...                       CENTRAL_MIDFIELD
1     122887  ...                                    NaN
2     122910  ...                       DEFENSE_MIDFIELD
3     122955  ...                                    NaN
4     123005  ...                       DEFENSE_MIDFIELD
5     123008  ...                       DEFENSE_MIDFIELD
6     123029  ...                       DEFENSE_MIDFIELD
7     123054  ...  ATTACKING_MIDFIELD / CENTRAL_MIDFIELD
8     123108  ...                                    NaN
9     123127  ...                       CENTRAL_MIDFIELD
10    123128  ...                                    NaN

[11 rows x 6 columns]


## Identify missing contributions worth checking

Find cases with recorded relevant actions but absent aggregate KPIs for targeted event-export checks.

In [8]:
# Inspect action outcomes where a contribution KPI is absent.
metric_action_map = {
    "pxt_pass": ("PASS", "recorded_passes"),
    "pxt_dribble": ("DRIBBLE", "recorded_carries"),
    "pxt_shot": ("SHOT", "recorded_shots"),
}

missing_action_reviews = []

for metric, (action_type, count_column) in metric_action_map.items():

    # One row per affected player-position-match.
    affected_records = advanced_activity_records.loc[
        advanced_activity_records[metric].isna()
        & advanced_activity_records[count_column].gt(0),
        advanced_keys + ["minutes_played"],
    ]

    # Retrieve the relevant recorded events.
    relevant_events = affected_records.merge(
        season_events.loc[
            season_events["actionType"].eq(action_type),
            advanced_keys + ["id", "result", "action"],
        ],
        on=advanced_keys,
        how="inner",
        validate="one_to_many",
    )

    # Summarise outcomes and action subtypes.
    review = (
        relevant_events.groupby(
            advanced_keys + ["result", "action"],
            dropna=False,
        )
        .size()
        .reset_index(name="event_count")
    )

    review.insert(0, "missing_metric", metric)
    missing_action_reviews.append(review)

missing_action_review = (
    pd.concat(missing_action_reviews, ignore_index=True)
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

display(
    missing_action_review[
        [
            "missing_metric",
            "match_id",
            "player_name",
            "player.position",
            "result",
            "action",
            "event_count",
        ]
    ]
    .sort_values(["missing_metric", "match_id", "player_name"])
    .reset_index(drop=True)
)

   missing_metric  match_id  ...            action event_count
0     pxt_dribble    122876  ...           DRIBBLE           1
1     pxt_dribble    122876  ...           DRIBBLE           2
2     pxt_dribble    122881  ...           DRIBBLE           1
3     pxt_dribble    122920  ...           DRIBBLE           3
4     pxt_dribble    122921  ...           DRIBBLE           1
5     pxt_dribble    122994  ...           DRIBBLE           1
6     pxt_dribble    122994  ...           DRIBBLE           2
7     pxt_dribble    123099  ...           DRIBBLE           3
8     pxt_dribble    123104  ...           DRIBBLE           1
9     pxt_dribble    123105  ...           DRIBBLE           1
10    pxt_dribble    123120  ...           DRIBBLE           1
11       pxt_pass    122879  ...          LOW_PASS           1
12       pxt_pass    122884  ...          LOW_PASS           1
13       pxt_shot    122844  ...            HEADER           1
14       pxt_shot    122865  ...    MID_RANGE_SHOT     

## Check event-level KPI exports

Read the targeted event-KPI files. The supplied snapshot provides two recoverable shooting values; other absent entries remain missing.

In [9]:
# One row for each affected player-position and missing metric.
cases_to_check = missing_action_review[
    advanced_keys + ["missing_metric"]
].drop_duplicates()

metric_to_kpi_id = {
    name: kpi_id
    for kpi_id, name in advanced_kpi_map.items()
}

event_kpi_folder = RAW_DIR / "events_kpis"
event_kpi_folder.mkdir(parents=True, exist_ok=True)

event_kpi_checks = []

for match_id in sorted(cases_to_check["match_id"].unique()):
    match_id = int(match_id)

    filename = f"events_kpis_{match_id}.json"
    url = f"{IMPECT_BASE_URL}/events_kpis/{filename}"

    # Reuse our existing download/cache function.
    event_kpis, status = load_or_download_json(
        url,
        event_kpi_folder / filename,
    )

    match_cases = cases_to_check.loc[
        cases_to_check["match_id"].eq(match_id)
    ]

    for _, case in match_cases.iterrows():
        metric = case["missing_metric"]
        kpi_id = metric_to_kpi_id[metric]

        values = [
            entry["value"]
            for entry in event_kpis
            if entry["playerId"] == case["player.id"]
            and entry["position"] == case["player.position"]
            and entry["kpiId"] == kpi_id
        ]

        event_kpi_checks.append({
            "match_id": match_id,
            "player.id": case["player.id"],
            "player.position": case["player.position"],
            "metric": metric,
            "event_kpi_entries": len(values),
            # No entry remains missing; do not assume zero here.
            "event_sum_when_present": (
                sum(values) if values else None
            ),
        })

    print(f"Checked match {match_id}: {status}")

event_kpi_reconciliation = (
    pd.DataFrame(event_kpi_checks)
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

display(
    event_kpi_reconciliation[
        [
            "player_name",
            "match_id",
            "player.position",
            "metric",
            "event_kpi_entries",
            "event_sum_when_present",
        ]
    ]
    .sort_values(["metric", "match_id"])
    .reset_index(drop=True)
)

Checked match 122844: cached
Checked match 122865: cached
Checked match 122871: cached
Checked match 122876: cached
Checked match 122879: cached
Checked match 122881: cached
Checked match 122884: cached
Checked match 122920: cached
Checked match 122921: cached
Checked match 122994: cached
Checked match 123001: cached
Checked match 123002: cached
Checked match 123055: cached
Checked match 123099: cached
Checked match 123104: cached
Checked match 123105: cached
Checked match 123120: cached
             player_name  match_id  ... event_kpi_entries event_sum_when_present
0           Felix Nmecha    122876  ...                 0                    NaN
1            Adrian Beck    122881  ...                 0                    NaN
2          Lucas Tousart    122920  ...                 0                    NaN
3            Rocco Reitz    122921  ...                 0                    NaN
4            Kevin Kampl    122994  ...                 0                    NaN
5         Aïssa Laïdo

## Keep value provenance and aggregate

Supplement only absent aggregates, preserve the raw provider values, and track whether each analysis value came from an aggregate or event export.

In [10]:
# Work on a copy; preserve the original extracted data.
advanced_analysis = advanced_records.copy()
metric_names = list(advanced_kpi_map.values())

# Track where each value comes from.
for metric in metric_names:
    source_column = f"{metric}_source"
    advanced_analysis[source_column] = "unresolved"
    advanced_analysis.loc[
        advanced_analysis[metric].notna(), source_column
    ] = "player_aggregate"

# Supplement absent aggregates with available event-export values.
event_supplements = event_kpi_reconciliation.loc[
    event_kpi_reconciliation["event_sum_when_present"].notna()
]

for _, row in event_supplements.iterrows():
    metric = row["metric"]

    mask = (
        advanced_analysis["match_id"].eq(row["match_id"])
        & advanced_analysis["player.id"].eq(row["player.id"])
        & advanced_analysis["player.position"].eq(row["player.position"])
    )

    if mask.sum() != 1:
        raise ValueError("Event supplement does not match exactly one record.")

    if advanced_analysis.loc[mask, metric].isna().all():
        advanced_analysis.loc[mask, metric] = row["event_sum_when_present"]
        advanced_analysis.loc[
            mask, f"{metric}_source"
        ] = "event_export"

# Build one summary row per player and metric.
summary_tables = []

for metric in metric_names:
    table = advanced_analysis[
        ["player.id", "minutes_played", metric, f"{metric}_source"]
    ].copy()

    table["minutes_with_value"] = table["minutes_played"].where(
        table[metric].notna(), 0
    )
    table["from_event_export"] = table[f"{metric}_source"].eq("event_export")

    summary = (
        table.groupby("player.id")
        .agg(
            available_value_sum=(
                metric, lambda values: values.sum(min_count=1)
            ),
            total_records=(metric, "size"),
            populated_records=(metric, "count"),
            midfield_minutes=("minutes_played", "sum"),
            minutes_with_value=("minutes_with_value", "sum"),
            supplemented_records=("from_event_export", "sum"),
        )
        .reset_index()
    )

    summary["metric"] = metric
    summary["unresolved_records"] = (
        summary["total_records"] - summary["populated_records"]
    )
    summary["minutes_with_value_pct"] = (
        summary["minutes_with_value"] / summary["midfield_minutes"] * 100
    )

    summary_tables.append(summary)

advanced_player_summary = (
    pd.concat(summary_tables, ignore_index=True)
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

# Preview passing pxT for 15 players, alphabetically.
display(
    advanced_player_summary.loc[
        advanced_player_summary["metric"].eq("pxt_pass"),
        [
            "player_name",
            "metric",
            "available_value_sum",
            "unresolved_records",
            "minutes_with_value_pct",
            "supplemented_records",
        ],
    ]
    .sort_values("player_name")
    .head(15)
    .round(4)
    .reset_index(drop=True)
)

            player_name    metric  ...  minutes_with_value_pct  supplemented_records
0           Adrian Beck  pxt_pass  ...                100.0000                     0
1   Aleksandar Pavlovic  pxt_pass  ...                100.0000                     0
2             Alex Král  pxt_pass  ...                 99.3885                     0
3        Amadou Haidara  pxt_pass  ...                100.0000                     0
4       Andrej Kramaric  pxt_pass  ...                100.0000                     0
5        András Schäfer  pxt_pass  ...                100.0000                     0
6        Angelo Stiller  pxt_pass  ...                100.0000                     0
7       Anthony Losilla  pxt_pass  ...                100.0000                     0
8           Anton Stach  pxt_pass  ...                100.0000                     0
9           Arne Engels  pxt_pass  ...                 99.3329                     0
10           Arne Maier  pxt_pass  ...                100.0000   

## Calculate coverage-aware advanced rates

A covered-period rate describes only the records with values; a full-season rate is shown only when all expected records have values. These KPIs do not change the core ranking.

In [11]:
# Rate over the playing-time records with available KPI values.
advanced_player_summary["observed_period_per90"] = (
    advanced_player_summary["available_value_sum"]
    / advanced_player_summary["minutes_with_value"].where(
        advanced_player_summary["minutes_with_value"].gt(0)
    )
    * 90
)

# Full-season rate only when every relevant record has a value.
all_records_present = (
    advanced_player_summary["unresolved_records"].eq(0)
)

advanced_player_summary["season_per90"] = (
    advanced_player_summary["available_value_sum"]
    / advanced_player_summary["midfield_minutes"]
    * 90
).where(all_records_present)

advanced_player_summary["data_status"] = "PARTIAL"
advanced_player_summary.loc[
    all_records_present, "data_status"
] = "ALL RECORDS PRESENT"

# Inspect familiar candidates across all six advanced metrics.
players_to_inspect = [
    "Xaver Schlager",
    "Granit Xhaka",
    "Anton Stach",
    "Angelo Stiller",
    "Florian Wirtz",
]

display(
    advanced_player_summary.loc[
        advanced_player_summary["player_name"].isin(players_to_inspect),
        [
            "player_name",
            "metric",
            "available_value_sum",
            "observed_period_per90",
            "season_per90",
            "minutes_with_value_pct",
            "unresolved_records",
            "data_status",
        ],
    ]
    .sort_values(["player_name", "metric"])
    .round(4)
    .reset_index(drop=True)
)

       player_name  ...          data_status
0   Angelo Stiller  ...  ALL RECORDS PRESENT
1   Angelo Stiller  ...              PARTIAL
2   Angelo Stiller  ...  ALL RECORDS PRESENT
3   Angelo Stiller  ...  ALL RECORDS PRESENT
4   Angelo Stiller  ...              PARTIAL
5   Angelo Stiller  ...              PARTIAL
6      Anton Stach  ...  ALL RECORDS PRESENT
7      Anton Stach  ...              PARTIAL
8      Anton Stach  ...              PARTIAL
9      Anton Stach  ...  ALL RECORDS PRESENT
10     Anton Stach  ...              PARTIAL
11     Anton Stach  ...              PARTIAL
12   Florian Wirtz  ...  ALL RECORDS PRESENT
13   Florian Wirtz  ...  ALL RECORDS PRESENT
14   Florian Wirtz  ...  ALL RECORDS PRESENT
15   Florian Wirtz  ...  ALL RECORDS PRESENT
16   Florian Wirtz  ...              PARTIAL
17   Florian Wirtz  ...              PARTIAL
18    Granit Xhaka  ...  ALL RECORDS PRESENT
19    Granit Xhaka  ...  ALL RECORDS PRESENT
20    Granit Xhaka  ...  ALL RECORDS PRESENT
21    Gran

## Prepare the advanced staging payload

Reshape 2,115 position-match records × six metrics into 12,690 rows, retaining raw and event-supplied values separately.

In [12]:
# Preserve the original values, before event supplementation.
advanced_kpis_to_load = advanced_records.melt(
    id_vars=advanced_keys,
    value_vars=list(advanced_kpi_map.values()),
    var_name="metric",
    value_name="player_aggregate_value",
)

metric_to_id = {
    name: kpi_id
    for kpi_id, name in advanced_kpi_map.items()
}

advanced_kpis_to_load["kpi_id"] = (
    advanced_kpis_to_load["metric"].map(metric_to_id)
)

# Keep independently retrieved event values in a separate column.
supplement_values = event_kpi_reconciliation[
    [
        "match_id",
        "player.id",
        "player.position",
        "metric",
        "event_sum_when_present",
    ]
].rename(columns={
    "event_sum_when_present": "event_export_value"
})

advanced_kpis_to_load = (
    advanced_kpis_to_load
    .merge(
        supplement_values,
        on=["match_id", "player.id", "player.position", "metric"],
        how="left",
        validate="many_to_one",
    )
    .rename(columns={
        "squadId": "team_id",
        "player.id": "player_id",
        "player.position": "position_name",
    })
)

advanced_kpis_to_load = advanced_kpis_to_load[
    [
        "match_id",
        "team_id",
        "player_id",
        "position_name",
        "kpi_id",
        "player_aggregate_value",
        "event_export_value",
    ]
]

advanced_sql_keys = [
    "match_id", "team_id", "player_id", "position_name", "kpi_id"
]

if advanced_kpis_to_load[advanced_sql_keys].isna().any().any():
    raise ValueError("Missing identifiers in advanced KPI records.")

if advanced_kpis_to_load.duplicated(advanced_sql_keys).any():
    raise ValueError("Duplicate advanced KPI records.")

expected_kpi_rows = len(advanced_records) * len(advanced_kpi_map)

if len(advanced_kpis_to_load) != expected_kpi_rows:
    raise ValueError("Unexpected advanced KPI row count.")

print("Advanced KPI rows ready:", len(advanced_kpis_to_load))
print(
    "Event-supplied values:",
    advanced_kpis_to_load["event_export_value"].notna().sum(),
)

display(advanced_kpis_to_load.head())

Advanced KPI rows ready: 12690
Event-supplied values: 2
   match_id  team_id  ...  player_aggregate_value event_export_value
0    122838       33  ...                    76.4                NaN
1    122838       33  ...                    12.7                NaN
2    122838       33  ...                    30.9                NaN
3    122838       33  ...                    23.3                NaN
4    122838       38  ...                    17.9                NaN

[5 rows x 7 columns]


## Save outputs

Save named intermediate tables for the next notebook. Generated files stay local and are excluded from Git.

In [13]:
assert len(advanced_kpis_to_load) == 12690
assert len(advanced_player_summary) == 474
assert int(advanced_kpis_to_load["event_export_value"].notna().sum()) == 2
save_context("advanced_analysis", globals(), ADVANCED_VARIABLES)
advanced_coverage.to_csv(REPORTS_DIR / "advanced_kpi_coverage.csv", index=False)
advanced_player_summary.to_csv(REPORTS_DIR / "advanced_player_summary.csv", index=False)


Saved local checkpoint: advanced_analysis.pkl
